In [13]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

print("Libraries ready")

Libraries ready


Generate dim_campaigns:

In [14]:
np.random.seed(42)
random.seed(42)

N_USERS = 5000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 6, 30)

campaigns = pd.DataFrame({
    'Campaign_ID': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006'],
    'Campaign_Name': [
        'Google Ads — Search',
        'Meta Ads — Social',
        'Influencer Partnership',
        'App Store Optimisation',
        'Email Referral',
        'TikTok Ads'
    ],
    'Channel': ['Paid Search', 'Paid Social', 'Influencer', 'Organic', 'Referral', 'Paid Social'],
    'Cost_USD': [45000, 38000, 25000, 0, 5000, 32000],
    'Target_Audience': ['25-34 Professional', '18-24 Student', '18-30 Lifestyle',
                        'All', 'Existing Users', '16-24 Gen Z']
})

print(campaigns)
campaigns.to_csv('dim_campaigns.csv', index=False)
print("dim_campaigns saved")

  Campaign_ID           Campaign_Name      Channel  Cost_USD  \
0        C001     Google Ads — Search  Paid Search     45000   
1        C002       Meta Ads — Social  Paid Social     38000   
2        C003  Influencer Partnership   Influencer     25000   
3        C004  App Store Optimisation      Organic         0   
4        C005          Email Referral     Referral      5000   
5        C006              TikTok Ads  Paid Social     32000   

      Target_Audience  
0  25-34 Professional  
1       18-24 Student  
2     18-30 Lifestyle  
3                 All  
4      Existing Users  
5         16-24 Gen Z  
dim_campaigns saved


Generate dim_users:

In [15]:
devices = ['iOS', 'Android']
countries = ['UK', 'USA', 'Canada', 'Australia', 'Germany', 'France', 'India', 'Singapore']
age_groups = ['18-24', '25-34', '35-44', '45-54', '55+']
app_versions = ['v2.1', 'v2.2', 'v2.3', 'v2.4']

campaign_weights = [0.25, 0.22, 0.12, 0.18, 0.10, 0.13]

users = []
for i in range(N_USERS):
    user_id = f'U{str(i+1).zfill(5)}'
    install_date = START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )
    campaign = random.choices(campaigns['Campaign_ID'].tolist(),
                              weights=campaign_weights)[0]
    device = random.choices(devices, weights=[0.58, 0.42])[0]
    country = random.choices(countries,
                             weights=[0.25, 0.30, 0.10, 0.08, 0.08, 0.07, 0.07, 0.05])[0]
    age_group = random.choices(age_groups,
                               weights=[0.25, 0.32, 0.22, 0.13, 0.08])[0]
    app_version = random.choices(app_versions,
                                 weights=[0.10, 0.20, 0.35, 0.35])[0]

    users.append({
        'User_ID': user_id,
        'Install_Date': install_date.strftime('%Y-%m-%d'),
        'Campaign_ID': campaign,
        'Device': device,
        'Country': country,
        'Age_Group': age_group,
        'App_Version': app_version
    })

df_users = pd.DataFrame(users)
print(f"Users generated: {len(df_users):,}")
print(df_users.head())
df_users.to_csv('dim_users.csv', index=False)
print("dim_users saved")

Users generated: 5,000
  User_ID Install_Date Campaign_ID   Device    Country Age_Group App_Version
0  U00001   2024-06-12        C001  Android         UK     18-24        v2.2
1  U00002   2024-05-19        C001      iOS         UK     18-24        v2.3
2  U00003   2024-01-07        C003  Android  Australia     25-34        v2.3
3  U00004   2024-03-12        C005      iOS    Germany     35-44        v2.3
4  U00005   2024-02-09        C001  Android         UK     25-34        v2.3
dim_users saved


Generate fact_events:

In [16]:
# Conversion probabilities per funnel stage
# These vary by campaign, device, age group to create interesting patterns
base_conversion = {
    'Signup':           0.72,
    'Onboarding':       0.55,
    'First_Action':     0.42,
    'Day1_Active':      0.38,
    'Day7_Active':      0.28
}

# Multipliers by campaign
campaign_conv_mult = {
    'C001': 1.15,  # Google Search — high intent
    'C002': 0.90,  # Meta Social — lower intent
    'C003': 0.95,  # Influencer
    'C004': 1.25,  # Organic — highest intent
    'C005': 1.20,  # Referral — high trust
    'C006': 0.80   # TikTok — lowest intent
}

# Multipliers by device
device_conv_mult = {'iOS': 1.10, 'Android': 0.92}

# Multipliers by age
age_conv_mult = {
    '18-24': 0.88,
    '25-34': 1.15,
    '35-44': 1.10,
    '45-54': 0.95,
    '55+': 0.80
}

events = []
funnel_stages = ['Install', 'Signup', 'Onboarding', 'First_Action', 'Day1_Active', 'Day7_Active']

for _, user in df_users.iterrows():
    install_date = datetime.strptime(user['Install_Date'], '%Y-%m-%d')
    camp_mult = campaign_conv_mult[user['Campaign_ID']]
    dev_mult = device_conv_mult[user['Device']]
    age_mult = age_conv_mult[user['Age_Group']]

    # Install always happens
    events.append({
        'Event_ID': f"E{len(events)+1:07d}",
        'User_ID': user['User_ID'],
        'Event_Type': 'Install',
        'Event_Date': install_date.strftime('%Y-%m-%d'),
        'Days_Since_Install': 0
    })

    # Each subsequent stage depends on previous
    current_date = install_date
    converted = True

    stage_days = {
        'Signup': 0,
        'Onboarding': 1,
        'First_Action': 2,
        'Day1_Active': 1,
        'Day7_Active': 6
    }

    for stage in funnel_stages[1:]:
        if not converted:
            break
        prob = base_conversion[stage] * camp_mult * dev_mult * age_mult
        prob = min(max(prob, 0.05), 0.95)

        if random.random() < prob:
            current_date = current_date + timedelta(days=stage_days[stage])
            events.append({
                'Event_ID': f"E{len(events)+1:07d}",
                'User_ID': user['User_ID'],
                'Event_Type': stage,
                'Event_Date': current_date.strftime('%Y-%m-%d'),
                'Days_Since_Install': (current_date - install_date).days
            })
        else:
            converted = False

df_events = pd.DataFrame(events)
print(f"Events generated: {len(df_events):,}")
print(f"Unique users with events: {df_events['User_ID'].nunique():,}")
print(f"\nEvent type counts:")
print(df_events['Event_Type'].value_counts())
df_events.to_csv('fact_events.csv', index=False)
print("\nfact_events saved")

Events generated: 13,092
Unique users with events: 5,000

Event type counts:
Event_Type
Install         5000
Signup          3831
Onboarding      2407
First_Action    1146
Day1_Active      515
Day7_Active      193
Name: count, dtype: int64

fact_events saved


Build funnel summary and validate:

In [17]:
# Count users at each stage
funnel_counts = df_events.groupby('Event_Type')['User_ID'].nunique()
stage_order = ['Install', 'Signup', 'Onboarding', 'First_Action', 'Day1_Active', 'Day7_Active']
funnel_counts = funnel_counts.reindex(stage_order)

print("=== FUNNEL SUMMARY ===\n")
prev = None
for stage, count in funnel_counts.items():
    if prev is None:
        print(f"{stage:20s}: {count:,} users (100%)")
    else:
        pct_of_install = count / funnel_counts['Install'] * 100
        pct_of_prev = count / prev * 100
        print(f"{stage:20s}: {count:,} users ({pct_of_install:.1f}% of installs | {pct_of_prev:.1f}% of previous stage)")
    prev = count

print(f"\nOverall Install → Day7 Active conversion: {funnel_counts['Day7_Active']/funnel_counts['Install']*100:.1f}%")

# Save funnel summary for Excel
funnel_summary = pd.DataFrame({
    'Stage': stage_order,
    'Users': funnel_counts.values,
    'Pct_of_Install': (funnel_counts.values / funnel_counts['Install'] * 100).round(1)
})
funnel_summary.to_csv('funnel_summary.csv', index=False)
print("\nFunnel summary saved")

=== FUNNEL SUMMARY ===

Install             : 5,000 users (100%)
Signup              : 3,831 users (76.6% of installs | 76.6% of previous stage)
Onboarding          : 2,407 users (48.1% of installs | 62.8% of previous stage)
First_Action        : 1,146 users (22.9% of installs | 47.6% of previous stage)
Day1_Active         : 515 users (10.3% of installs | 44.9% of previous stage)
Day7_Active         : 193 users (3.9% of installs | 37.5% of previous stage)

Overall Install → Day7 Active conversion: 3.9%

Funnel summary saved


Export Excel-ready files:

In [18]:
print("=== EXPORTING EXCEL FILES ===\n")

# Main files for Power Query
df_users.to_excel('dim_users.xlsx', index=False)
df_events.to_excel('fact_events.xlsx', index=False)
campaigns.to_excel('dim_campaigns.xlsx', index=False)
funnel_summary.to_excel('funnel_summary.xlsx', index=False)

# Conversion by campaign
conv_campaign = df_events[df_events['Event_Type']=='Day7_Active']['User_ID'].tolist()
df_users['Converted_Day7'] = df_users['User_ID'].isin(conv_campaign).astype(int)
camp_conv = df_users.groupby('Campaign_ID').agg(
    Total_Users=('User_ID', 'count'),
    Converted=('Converted_Day7', 'sum')
).reset_index()
camp_conv['Conversion_Rate'] = (camp_conv['Converted'] / camp_conv['Total_Users'] * 100).round(1)
camp_conv = camp_conv.merge(campaigns[['Campaign_ID', 'Campaign_Name', 'Channel', 'Cost_USD']], on='Campaign_ID')
camp_conv['Cost_Per_Acquisition'] = (camp_conv['Cost_USD'] / camp_conv['Converted'].replace(0, 1)).round(2)
camp_conv.to_excel('campaign_performance.xlsx', index=False)

print("✓ dim_users.xlsx")
print("✓ fact_events.xlsx")
print("✓ dim_campaigns.xlsx")
print("✓ funnel_summary.xlsx")
print("✓ campaign_performance.xlsx")
print("\nAll files ready for Excel Power Query")

=== EXPORTING EXCEL FILES ===

✓ dim_users.xlsx
✓ fact_events.xlsx
✓ dim_campaigns.xlsx
✓ funnel_summary.xlsx
✓ campaign_performance.xlsx

All files ready for Excel Power Query
